In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
import glob
import re
import os
from pathlib import Path
%matplotlib inline 
import matplotlib.ticker as ticker
from IPython.display import display, HTML
from ipywidgets import interact


# Reading in Dataframe and Dataframe preparation

In [2]:
all_months_df_raw = pd.read_excel("rptMealsServedBydayBySchool_ALLMONTHS2024-2025.xlsx")

In [3]:
# remove the total rows
all_months_df = all_months_df_raw[all_months_df_raw.iloc[:, 1] != 0]

all_months_df = all_months_df.copy()

#all_months_df.to_excel("allmonths_2024-2025_cleaned.xlsx", index=False)

# cleaning and adding new cols and sorting
all_months_df['Date'] = all_months_df['Date'].replace('February 2024', 'February 2025')
all_months_df['%Free/Reduced'] = all_months_df['Eco Dis'] / all_months_df['Enrolled']

# sort dates chronologically
all_months_df['Date_dt'] = pd.to_datetime(all_months_df['Date'], format='%B %Y')
all_months_df = all_months_df.sort_values('Date_dt')

In [4]:
all_months_df['Date'].unique()

array(['July 2024', 'August 2024', 'September 2024', 'October 2024',
       'November 2024', 'December 2024', 'January 2025', 'February 2025',
       'March 2025', 'April 2025', 'May 2025', 'June 2025'], dtype=object)

In [5]:
merged_df = pd.read_excel("all_months_bf_serving_merged.xlsx")

# Eco Dis vs ADP by Site

## Code

### Analysis Code

In [6]:
def plot_site_eco_adp(sitenumber):
    # get % eco dis and % participation dictionaries to plot
    monthly_percent_ecodis = {}
    monthly_percent_adp = {}
    pos_setup = ''
    for sitenum, site_df in merged_df.groupby('site', sort=False):
        #print(sitenum, site_df)
        if str(sitenum) == sitenumber:
            print('Yes')
            pos_setup = site_df['POS Setup'].iloc[0]
            for month, month_df in site_df.groupby('Date', sort=False):
                monthly_percent_ecodis[month] = month_df['%Free/Reduced'].iloc[0]
                monthly_percent_adp[month] = month_df['%Participation vs Enroll'].iloc[0]
        else:
            continue

    # plot %eco dis and %participation by month
    months = list(monthly_percent_adp.keys())
    monthly_adps = list(monthly_percent_adp.values())
    monthly_ecodis = list(monthly_percent_ecodis.values())

    x = np.arange(len(months))
    bar_width = 0.4

    fig, ax = plt.subplots(figsize=(10,6))

    adp_bars = ax.bar(x - bar_width/2, monthly_adps, bar_width, label='Total ADP', color='skyblue')
    ecodis_bars = ax.bar(x + bar_width/2, monthly_ecodis, bar_width, label='Total Eco Dis', color='orange')

    ax.yaxis.set_major_formatter(PercentFormatter(1.0))

    ax.set_ylabel('% Meal Participation and Eco Dis of Enrollment')
    ax.set_xlabel('Month')
    ax.set_title('Percent ADP and Eco Dis Students 2024-2025, Site: ' + sitenumber)
    ax.set_xticks(x)
    ax.set_xticklabels(months, rotation=45)
    ax.legend()

    plt.tight_layout()
    plt.grid(True, linestyle='--', alpha=0.7);
    plt.show()
    print("POS Setup: " + pos_setup)


### Plot Code

In [7]:
interact(plot_site_eco_adp, sitenumber='1186')

interactive(children=(Text(value='1186', description='sitenumber'), Output()), _dom_classes=('widget-interact'…

<function __main__.plot_site_eco_adp(sitenumber)>

# By Site Analyses

## Code

### Analysis Code

In [8]:
def plot_site_var(sitenumber, var):
    # get the specificed monthly data into a dict to plot
    monthly_var_dict = {}
    for sitenum, site_df in all_months_df.groupby('site', sort=False):
        if str(sitenum) == sitenumber:
            for month, month_df in site_df.groupby('Date', sort=False):
                monthly_var_dict[month] = month_df[var].iloc[0]
        else:
            continue
    # plot
    months = list(monthly_var_dict.keys())
    monthly_var = list(monthly_var_dict.values())

    fig, ax = plt.subplots(figsize=(10,6))
    x = np.arange(len(months))

    if var == '%Participation vs Enroll':
        monthly_var = [x * 100 for x in monthly_var]

    ax.plot(x, monthly_var, marker='o', label=var)
    
    if var == '%Participation vs Enroll':
        ax.yaxis.set_major_formatter(PercentFormatter(100.0))

    for i, value in enumerate(monthly_var):
        # ax.text(x_position, y_position, text_string)
        value = value.round(1)
        ax.text(x[i], value, str(value), 
                ha='center',    # Centers the text horizontally over the point
                va='bottom',    # Places the text just above the point
                fontsize=10)     # Slightly smaller font so it doesn't clutter

    

    ax.set_ylabel(var)
    ax.set_xlabel('Month')
    ax.set_title(var + ' 2024-2025, Site: ' + sitenumber)
    ax.set_xticks(x)
    ax.set_xticklabels(months, rotation=45)
    ax.legend()

    ax.set_ylim(bottom=0, top=max(monthly_var)*1.1)
    plt.tight_layout()
    plt.grid(True, linestyle='--', alpha=0.7);
    plt.show()

### Plot Code

In [9]:
interact(plot_site_var, sitenumber='1186', var='Days Served')

interactive(children=(Text(value='1186', description='sitenumber'), Text(value='Days Served', description='var…

<function __main__.plot_site_var(sitenumber, var)>